# Cybersecurity Mean-Field Control
## Simplex, logit, and adaptive MF-REINFORCE

This notebook completes the cybersecurity numerical experiment using the same structure as `twostate_twoaction.ipynb`.
The model follows the cybersecurity benchmark described in the research project notes: states are `DI`, `DS`, `UI`, and `US`, actions either keep the current defense status or trigger a protection-status switch, and infection intensities depend nonlinearly on the current population law.

The notebook uses exact population recursion for validation in every experiment. Training can use either exact population flow or a particle-estimated flow, depending on the scenario.

### Experiments

1. **Equal-parameter exact flow.** Simplex and logits use the same Monte Carlo settings, `(B, n) = (200, 10)`.
2. **Equal-budget exact flow.** Logits uses `(B, n) = (200, 10)`. Simplex uses `(B, n) = (3675, 525)`, giving both methods `12600` simulated transitions per update for `T_train=3`.
3. **Equal-budget estimated flow.** The same budgets are used with a particle flow estimate using `flow_particles=200`, giving both methods `13200` simulated transitions per update.
4. **Adaptive exact-flow comparison.** Finite-budget and consistent adaptive simplex methods are compared with the best fixed simplex and best fixed logits configurations from the equal-budget exact-flow experiment.

The full experiment cells are intentionally output-free in the saved notebook. They are ready to run, but they are not pre-executed.


## Imports


In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple
import copy
import math
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm import tqdm

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = (ROOT / "..").resolve()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from mfc.algorithms import (
    AdaptiveSimplexControllerConfig,
    ConsistentAdaptiveSimplexMFREINFORCE,
    FiniteBudgetAdaptiveSimplexMFREINFORCE,
    LogitsPerturbedMFREINFORCE,
    SimplexPerturbedMFREINFORCE,
)
from mfc.environments import CybersecurityConfig, CybersecurityMFC, CybersecurityPolicy


In [2]:
### DTYPE SET TO TORCH32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
torch.set_default_dtype(DTYPE)

## Runtime And General Helpers


In [3]:
def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)


set_seed(0)
print(f"device: {device}")


def format_runtime(seconds: Optional[float]) -> str:
    if seconds is None:
        return "not set"
    seconds = float(seconds)
    hours, remainder = divmod(int(seconds), 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f"{hours:d}h {minutes:02d}m {secs:02d}s"
    if minutes:
        return f"{minutes:d}m {secs:02d}s"
    return f"{seconds:.1f}s"


def std_ddof(n: int) -> int:
    return 1 if n > 1 else 0


def safe_name(name: object) -> str:
    text = str(name)
    return "".join(ch.lower() if ch.isalnum() else "_" for ch in text).strip("_") or "cybersecurity"


def tensor_float(value, default: float = float("nan")) -> float:
    if value is None:
        return default
    if isinstance(value, torch.Tensor):
        if value.numel() != 1:
            return default
        return float(value.detach().cpu().item())
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def aligned_validation_history(runs):
    min_len = min(len(run["history"]["validation_value"]) for run in runs)
    episodes = np.asarray(runs[0]["history"]["episode"][:min_len], dtype=float)
    values = np.asarray([run["history"]["validation_value"][:min_len] for run in runs], dtype=float)
    return episodes, values


def aligned_history_metric(runs, key: str):
    lengths = [len(run["history"].get(key, [])) for run in runs]
    if not lengths or min(lengths) == 0:
        return np.asarray([]), np.asarray([])
    min_len = min(lengths)
    episodes = np.asarray(runs[0]["history"]["episode"][:min_len], dtype=float)
    values = np.asarray([run["history"][key][:min_len] for run in runs], dtype=float)
    return episodes, values


device: cuda


## Policy, Run Plan, And Flow Helpers


In [4]:
def fixed_validation_law(config: CybersecurityConfig) -> torch.Tensor:
    return torch.full(
        (config.n_states,),
        1.0 / config.n_states,
        dtype=config.dtype,
        device=config.device,
    )


def sample_cybersecurity_initial_laws(config: CybersecurityConfig, count: int) -> torch.Tensor:
    concentration = torch.ones(config.n_states, dtype=config.dtype, device=config.device)
    return torch.distributions.Dirichlet(concentration).sample((count,)).detach().cpu()


def clone_state_dict(policy: CybersecurityPolicy) -> Dict[str, torch.Tensor]:
    return {key: value.detach().cpu().clone() for key, value in policy.state_dict().items()}


def prepare_paired_run_plans(
    config: CybersecurityConfig,
    seed_base: int = 51_000,
    training_runs: Optional[int] = None,
) -> List[Dict[str, object]]:
    n_runs = config.training_runs if training_runs is None else int(training_runs)
    plans: List[Dict[str, object]] = []
    for run_idx in range(n_runs):
        seed = seed_base + run_idx
        set_seed(seed)
        policy = CybersecurityPolicy(config)
        plans.append(
            {
                "run_idx": run_idx,
                "seed": seed,
                "initial_control": {"state_dict": clone_state_dict(policy)},
                "initial_laws": sample_cybersecurity_initial_laws(config, config.n_train),
            }
        )
    return plans


def load_policy(config: CybersecurityConfig, payload: Dict[str, object], trainable: bool = True) -> CybersecurityPolicy:
    policy = CybersecurityPolicy(config)
    state = {
        key: value.to(dtype=config.dtype, device=config.device).detach().clone()
        for key, value in payload["state_dict"].items()
    }
    policy.load_state_dict(state)
    policy.train(trainable)
    for parameter in policy.parameters():
        parameter.requires_grad_(trainable)
    return policy


def payload_from_record(record: Dict[str, object]) -> Dict[str, object]:
    return {"state_dict": {key: value.detach().cpu().clone() for key, value in record["policy_state_dict"].items()}}


def parameter_vector(policy: CybersecurityPolicy) -> torch.Tensor:
    return torch.nn.utils.parameters_to_vector(policy.parameters()).detach()


def assign_flat_ascent_gradient(policy: CybersecurityPolicy, grad_hat: torch.Tensor) -> None:
    grad_flat = grad_hat.detach().reshape(-1)
    offset = 0
    for parameter in policy.parameters():
        count = parameter.numel()
        parameter.grad = -grad_flat[offset : offset + count].reshape_as(parameter).clone()
        offset += count
    if offset != grad_flat.numel():
        raise ValueError(f"Gradient length {grad_flat.numel()} does not match policy parameter length {offset}.")


@torch.no_grad()
def exact_population_flow_batch(
    env: CybersecurityMFC,
    policy: CybersecurityPolicy,
    mu0_batch: torch.Tensor,
    horizon: int,
) -> torch.Tensor:
    flow = [mu0_batch]
    for t in range(horizon):
        kernel = env.averaged_kernel(policy, t, flow[-1])
        flow.append(torch.einsum("bi,bij->bj", flow[-1], kernel))
    return torch.stack(flow, dim=1)


def training_population_flow(
    env: CybersecurityMFC,
    algorithm,
    policy: CybersecurityPolicy,
    mu0: torch.Tensor,
    horizon: int,
    flow_mode: str,
    flow_particles: int,
) -> torch.Tensor:
    if flow_mode == "exact":
        with torch.no_grad():
            return env.exact_population_flow(policy, mu0, horizon).detach()
    if flow_mode == "particle":
        return algorithm.estimate_population_flow(policy, mu0, flow_particles, horizon=horizon).detach()
    raise ValueError(f"Unknown flow_mode={flow_mode!r}.")


## Metrics, Costs, And Perturbation Calibration


In [5]:
def reference_metrics(
    env: CybersecurityMFC,
    policy: CybersecurityPolicy,
    mu0: torch.Tensor,
    horizon: int,
) -> Dict[str, object]:
    was_training = policy.training
    policy.eval()
    try:
        with torch.no_grad():
            flow = env.exact_population_flow(policy, mu0, horizon).detach()
            value = env.exact_value(policy, mu0, horizon).detach()
            infected = flow[:, env.config.DI] + flow[:, env.config.UI]
            defended = flow[:, env.config.DI] + flow[:, env.config.DS]
            update_probs = []
            for t in range(horizon):
                pi = env.action_probabilities(policy, t, flow[t])
                update_probs.append((flow[t] * pi[:, env.config.UPDATE]).sum())
            update_probs_t = torch.stack(update_probs) if update_probs else torch.zeros(0, dtype=flow.dtype, device=flow.device)
    finally:
        policy.train(was_training)

    final_distribution = flow[-1].detach().cpu()
    state_names = env.config.cyber_state_names
    metrics: Dict[str, object] = {
        "value": float(value.item()),
        "flow": flow.detach().cpu(),
        "infected": infected.detach().cpu(),
        "defended": defended.detach().cpu(),
        "update_probability": update_probs_t.detach().cpu(),
        "final_distribution": final_distribution,
        "terminal_infected": float(infected[-1].item()),
        "terminal_defended": float(defended[-1].item()),
        "mean_infected": float(infected.mean().item()),
        "mean_defended": float(defended.mean().item()),
        "mean_update_probability": float(update_probs_t.mean().item()) if update_probs_t.numel() else float("nan"),
    }
    for idx, state_name in enumerate(state_names):
        metrics[f"terminal_{state_name}"] = float(final_distribution[idx].item())
    return metrics


def record_numeric_history(history: Dict[str, List[float]], metrics: Dict[str, object]) -> None:
    for key, value in metrics.items():
        if isinstance(value, (int, float, np.integer, np.floating)):
            history.setdefault(key, []).append(float(value))


def simulator_transitions_per_update(
    algorithm_name: str,
    horizon: int,
    B: int,
    n_aux_or_inner: int,
    flow_mode: str = "exact",
    flow_particles: int = 0,
    diagnostic_replications: int = 0,
    checkpoint_interval: int = 0,
) -> int:
    normalized = algorithm_name.lower().replace("-", "").replace("_", "").replace(" ", "")
    if normalized in {"simplex", "fixedsimplex"}:
        core = (int(B) + int(n_aux_or_inner)) * int(horizon)
    elif normalized in {"finiteadaptivesimplex", "finitebudgetadaptivesimplex", "consistentadaptivesimplex", "adaptivesimplex"}:
        core = (int(B) + int(n_aux_or_inner)) * int(horizon)
        if diagnostic_replications > 0 and checkpoint_interval > 0:
            core = core * (1.0 + 2.0 * int(diagnostic_replications) / int(checkpoint_interval))
    elif normalized == "logits":
        core = int(B) * int(horizon) + int(B) * int(n_aux_or_inner) * int(horizon) * (int(horizon) + 1) // 2
    else:
        raise ValueError(f"Unknown algorithm_name={algorithm_name!r}.")
    flow_cost = 0 if flow_mode == "exact" else int(flow_particles) * int(horizon)
    return int(math.ceil(core + flow_cost))


def simplex_budget_matching_logit(
    horizon: int,
    logit_B: int = 200,
    logit_n: int = 10,
    simplex_aux_fraction: float = 0.125,
) -> Dict[str, Dict[str, int]]:
    simplex_total_paths = logit_B + logit_B * logit_n * (horizon + 1) // 2
    simplex_n = int(round(simplex_total_paths * simplex_aux_fraction))
    simplex_B = int(simplex_total_paths - simplex_n)
    return {
        "Simplex": {"B": simplex_B, "n": simplex_n},
        "Logits": {"B": int(logit_B), "n": int(logit_n)},
    }


def simulator_cost_table(
    horizon: int,
    flow_mode: str,
    budgets: Dict[str, Dict[str, int]],
    flow_particles: int = 0,
) -> pd.DataFrame:
    rows = []
    simplex_cost = simulator_transitions_per_update(
        "Simplex",
        horizon,
        budgets["Simplex"]["B"],
        budgets["Simplex"]["n"],
        flow_mode=flow_mode,
        flow_particles=flow_particles,
    )
    for algorithm_name in ["Simplex", "Logits"]:
        budget = budgets[algorithm_name]
        cost = simulator_transitions_per_update(
            algorithm_name,
            horizon,
            budget["B"],
            budget["n"],
            flow_mode=flow_mode,
            flow_particles=flow_particles,
        )
        rows.append(
            {
                "algorithm": algorithm_name,
                "B": budget["B"],
                "n": budget["n"],
                "horizon": horizon,
                "flow_mode": flow_mode,
                "flow_particles": flow_particles if flow_mode == "particle" else 0,
                "simulator_transitions_per_update": cost,
                "ratio_vs_simplex": cost / simplex_cost,
            }
        )
    return pd.DataFrame(rows).set_index("algorithm")


def adaptive_cost_table(
    horizon: int,
    flow_mode: str,
    budgets: Dict[str, Dict[str, int]],
    controller_config: AdaptiveSimplexControllerConfig,
    flow_particles: int = 0,
) -> pd.DataFrame:
    rows = []
    for algorithm_name, budget_key in [
        ("Simplex", "Simplex"),
        ("Logits", "Logits"),
        ("Finite Adaptive Simplex", "Simplex"),
        ("Consistent Adaptive Simplex", "Simplex"),
    ]:
        budget = budgets[budget_key]
        kwargs = {}
        if "Adaptive" in algorithm_name:
            kwargs = {
                "diagnostic_replications": controller_config.diagnostic_replications,
                "checkpoint_interval": controller_config.checkpoint_interval,
            }
        cost = simulator_transitions_per_update(
            algorithm_name,
            horizon,
            budget["B"],
            budget["n"],
            flow_mode=flow_mode,
            flow_particles=flow_particles,
            **kwargs,
        )
        base_cost = simulator_transitions_per_update(
            "Simplex" if algorithm_name != "Logits" else "Logits",
            horizon,
            budget["B"],
            budget["n"],
            flow_mode=flow_mode,
            flow_particles=flow_particles,
        )
        rows.append(
            {
                "algorithm": algorithm_name,
                "B": budget["B"],
                "n": budget["n"],
                "base_transitions_per_update": base_cost,
                "expected_transitions_per_update": cost,
                "controller_extra_reported_separately": bool("Adaptive" in algorithm_name),
            }
        )
    return pd.DataFrame(rows).set_index("algorithm")


def sample_simplex_q(config: CybersecurityConfig, n: int) -> torch.Tensor:
    u = config.q_sigma * torch.randn(n, config.n_states - 1, dtype=config.dtype, device=config.device)
    logits = torch.cat([u, torch.zeros(n, 1, dtype=config.dtype, device=config.device)], dim=-1)
    q = torch.softmax(logits, dim=-1).clamp_min(config.q_clip)
    return q / q.sum(dim=-1, keepdim=True)


def collect_reference_laws(
    config: CybersecurityConfig,
    payload: Dict[str, object],
    count: int,
    horizon: int,
    chunk_size: int = 512,
) -> torch.Tensor:
    env = CybersecurityMFC(config)
    policy = load_policy(config, payload, trainable=False)
    initial_laws = sample_cybersecurity_initial_laws(config, count).to(dtype=config.dtype, device=config.device)
    chunks = []
    for start in range(0, count, chunk_size):
        mu0_batch = initial_laws[start : start + chunk_size]
        flow = exact_population_flow_batch(env, policy, mu0_batch, horizon)
        chunks.append(flow.reshape(-1, config.n_states).detach().cpu())
    return torch.cat(chunks, dim=0)


def calibrate_perturbation_radii(
    config: CybersecurityConfig,
    reference_laws: torch.Tensor,
    simplex_values: Sequence[float],
    logit_values: Sequence[float],
    num_samples: int = 20_000,
) -> pd.DataFrame:
    reference_laws = reference_laws.to(dtype=config.dtype, device=config.device)
    index = torch.randint(reference_laws.shape[0], (num_samples,), device=config.device)
    mu = reference_laws[index].clamp_min(config.q_clip)
    mu = mu / mu.sum(dim=-1, keepdim=True)
    q = sample_simplex_q(config, num_samples)
    noise = torch.randn(num_samples, config.n_states, dtype=config.dtype, device=config.device)
    log_mu = torch.log(mu)
    rows = []

    for value in simplex_values:
        perturbed = (1.0 - float(value)) * mu + float(value) * q
        radii = 0.5 * (mu - perturbed).abs().sum(dim=-1).detach().cpu().numpy()
        rows.append(
            {
                "algorithm": "Simplex",
                "parameter": float(value),
                "tv_mean": radii.mean(),
                "tv_median": np.median(radii),
                "tv_std": radii.std(ddof=0),
                "tv_p10": np.quantile(radii, 0.10),
                "tv_p90": np.quantile(radii, 0.90),
            }
        )

    for value in logit_values:
        perturbed = torch.softmax(log_mu + float(value) * noise, dim=-1)
        radii = 0.5 * (mu - perturbed).abs().sum(dim=-1).detach().cpu().numpy()
        rows.append(
            {
                "algorithm": "Logits",
                "parameter": float(value),
                "tv_mean": radii.mean(),
                "tv_median": np.median(radii),
                "tv_std": radii.std(ddof=0),
                "tv_p10": np.quantile(radii, 0.10),
                "tv_p90": np.quantile(radii, 0.90),
            }
        )
    return pd.DataFrame(rows)


## Training Runners


In [6]:
ALGORITHM_LABELS = {
    "simplex": "Simplex",
    "logits": "Logits",
    "finite_adaptive": "Finite Adaptive Simplex",
    "consistent_adaptive": "Consistent Adaptive Simplex",
}


def finite_controller_for_initial_lambda(initial_lambda: float) -> AdaptiveSimplexControllerConfig:
    return AdaptiveSimplexControllerConfig(
        initial_lambda=float(initial_lambda),
        lambda_min=0.01,
        lambda_max=0.8,
        checkpoint_interval=100,
        diagnostic_replications=4,
        contraction=0.5,
    )


def consistent_controller_for_initial_lambda(initial_lambda: float) -> AdaptiveSimplexControllerConfig:
    return AdaptiveSimplexControllerConfig(
        initial_lambda=float(initial_lambda),
        lambda_min=0.01,
        lambda_max=0.8,
        checkpoint_interval=100,
        diagnostic_replications=4,
        contraction=0.5,
        envelope_lambda0=float(initial_lambda),
        envelope_m0=1000.0,
        envelope_zeta=0.25,
        eta_power=1.5,
        main_sample_growth_power=0.25,
        aux_sample_growth_power=0.5,
        sample_growth_interval=1000.0,
    )


def new_algorithm(method: str, env: CybersecurityMFC, initial_lambda: Optional[float] = None):
    if method == "simplex":
        return SimplexPerturbedMFREINFORCE(env)
    if method == "logits":
        return LogitsPerturbedMFREINFORCE(env)
    if method == "finite_adaptive":
        return FiniteBudgetAdaptiveSimplexMFREINFORCE(env, finite_controller_for_initial_lambda(float(initial_lambda)))
    if method == "consistent_adaptive":
        return ConsistentAdaptiveSimplexMFREINFORCE(env, consistent_controller_for_initial_lambda(float(initial_lambda)))
    raise ValueError(f"Unknown method={method!r}.")


def gradient_for_method(
    method: str,
    algorithm,
    policy: CybersecurityPolicy,
    mu0: torch.Tensor,
    mu_flow: torch.Tensor,
    episode: int,
    B: int,
    n_aux_or_inner: int,
    perturbation: Optional[float],
):
    horizon = mu_flow.shape[0] - 1
    if method == "simplex":
        value = float(perturbation)
        return algorithm.complete_gradient_estimate(
            policy,
            mu_flow,
            value,
            B,
            n_aux_or_inner,
            eta=value,
            baseline="batch_mean",
        )
    if method == "logits":
        value = float(perturbation)
        return algorithm.gradient_estimate(
            policy,
            mu0,
            value,
            B,
            n_aux_or_inner,
            flow_particles=1,
            horizon=horizon,
            mu_flow=mu_flow,
        )
    if method in {"finite_adaptive", "consistent_adaptive"}:
        return algorithm.gradient_estimate(policy, mu_flow, episode, B, n_aux_or_inner, baseline="batch_mean")
    raise ValueError(f"Unknown method={method!r}.")


def base_transition_cost(
    method: str,
    horizon: int,
    B: int,
    n_aux_or_inner: int,
    flow_mode: str,
    flow_particles: int,
) -> int:
    algorithm_name = ALGORITHM_LABELS[method]
    return simulator_transitions_per_update(
        algorithm_name,
        horizon,
        B,
        n_aux_or_inner,
        flow_mode=flow_mode,
        flow_particles=flow_particles,
    )


def train_cybersecurity_method(
    method: str,
    config: CybersecurityConfig,
    perturbation_values: Sequence[float],
    budget: Dict[str, int],
    run_plans: List[Dict[str, object]],
    flow_mode: str,
    flow_particles: int,
    train_horizon: int,
    validation_horizon: int,
    label: str,
    show_progress: bool = True,
    early_stopping_patience: Optional[int] = None,
    early_stopping_min_delta: float = 0.0,
    max_runtime_seconds: Optional[float] = None,
) -> Dict[float, List[Dict[str, object]]]:
    algorithm_name = ALGORITHM_LABELS[method]
    B = int(budget["B"])
    n_aux_or_inner = int(budget["n"])
    fixed_mu0 = fixed_validation_law(config)
    results: Dict[float, List[Dict[str, object]]] = {}

    for perturbation in perturbation_values:
        parameter_key = float(perturbation)
        results[parameter_key] = []
        for run_idx, plan in enumerate(run_plans):
            set_seed(int(plan["seed"]))
            env = CybersecurityMFC(config)
            policy = load_policy(config, plan["initial_control"], trainable=True)
            optimizer = torch.optim.Adam(policy.parameters(), lr=config.lr)
            algorithm = new_algorithm(method, env, initial_lambda=parameter_key)
            history: Dict[str, List[float]] = {
                "episode": [],
                "validation_value": [],
                "train_return_mean": [],
                "grad_norm": [],
                "lambda": [],
                "eta": [],
                "lambda_ctrl": [],
                "epsilon": [],
                "cumulative_simulator_transitions": [],
                "elapsed_seconds": [],
            }
            controller_records: List[Dict[str, object]] = []
            cumulative_transitions = 0
            run_start = time.perf_counter()
            stop_reason = "completed"
            episodes_completed = 0
            best_validation = -float("inf")
            best_episode = None
            best_state_dict = None
            checks_since_best = 0

            iterator = range(config.n_train)
            if show_progress:
                iterator = tqdm(iterator, desc=f"{label} {algorithm_name} p={parameter_key:g} run={run_idx}")

            for episode in iterator:
                if max_runtime_seconds is not None and time.perf_counter() - run_start >= max_runtime_seconds:
                    stop_reason = f"max_runtime_{format_runtime(max_runtime_seconds)}"
                    break

                mu0 = plan["initial_laws"][episode].to(dtype=config.dtype, device=config.device)
                mu_flow = training_population_flow(
                    env,
                    algorithm,
                    policy,
                    mu0,
                    train_horizon,
                    flow_mode,
                    flow_particles,
                )
                grad_hat, diag = gradient_for_method(
                    method,
                    algorithm,
                    policy,
                    mu0,
                    mu_flow,
                    episode,
                    B,
                    n_aux_or_inner,
                    None if method in {"finite_adaptive", "consistent_adaptive"} else parameter_key,
                )

                transition_value = tensor_float(diag.get("simulator_transitions"), default=float("nan"))
                if math.isnan(transition_value):
                    transitions = base_transition_cost(
                        method,
                        train_horizon,
                        B,
                        n_aux_or_inner,
                        flow_mode,
                        flow_particles,
                    )
                else:
                    transitions = int(transition_value)
                cumulative_transitions += transitions

                controller_diag = diag.get("controller")
                if controller_diag is not None:
                    controller_records.append({"episode": episode, **controller_diag})

                optimizer.zero_grad(set_to_none=True)
                assign_flat_ascent_gradient(policy, grad_hat)
                optimizer.step()
                episodes_completed = episode + 1

                if episode % config.validate_every == 0 or episode == config.n_train - 1:
                    metrics = reference_metrics(env, policy, fixed_mu0, validation_horizon)
                    validation_value = metrics["value"]
                    history["episode"].append(float(episode))
                    history["validation_value"].append(validation_value)
                    history["train_return_mean"].append(tensor_float(diag.get("mean_return")))
                    history["grad_norm"].append(tensor_float(diag.get("grad_norm")))
                    history["lambda"].append(tensor_float(diag.get("lambda"), default=float("nan")))
                    history["eta"].append(tensor_float(diag.get("eta"), default=float("nan")))
                    history["lambda_ctrl"].append(tensor_float(diag.get("lambda_ctrl"), default=float("nan")))
                    history["epsilon"].append(parameter_key if method == "logits" else float("nan"))
                    history["cumulative_simulator_transitions"].append(float(cumulative_transitions))
                    history["elapsed_seconds"].append(time.perf_counter() - run_start)
                    record_numeric_history(history, metrics)

                    if validation_value > best_validation + early_stopping_min_delta:
                        best_validation = validation_value
                        best_episode = episode
                        best_state_dict = clone_state_dict(policy)
                        checks_since_best = 0
                    else:
                        checks_since_best += 1

                    if show_progress:
                        iterator.set_postfix(value=f"{validation_value:.4g}", grad=f"{history['grad_norm'][-1]:.3g}")

                    if early_stopping_patience is not None and checks_since_best >= early_stopping_patience:
                        stop_reason = "early_stopping"
                        break

            final_metrics = reference_metrics(env, policy, fixed_mu0, validation_horizon)
            record = {
                "algorithm": algorithm_name,
                "method": method,
                "flow_mode": flow_mode,
                "flow_particles": flow_particles if flow_mode == "particle" else 0,
                "parameter": parameter_key,
                "run_idx": run_idx,
                "seed": int(plan["seed"]),
                "policy_state_dict": clone_state_dict(policy),
                "history": history,
                "controller_diagnostics": controller_records,
                "final_value": final_metrics["value"],
                "reference_metrics": final_metrics,
                "runtime_seconds": time.perf_counter() - run_start,
                "episodes_completed": episodes_completed,
                "main_trajectories": B,
                "auxiliary_trajectories": n_aux_or_inner,
                "simulator_transitions_per_update": base_transition_cost(
                    method,
                    train_horizon,
                    B,
                    n_aux_or_inner,
                    flow_mode,
                    flow_particles,
                ),
                "total_simulator_transitions": cumulative_transitions,
                "stop_reason": stop_reason,
                "best_validation_value": best_validation,
                "best_episode": best_episode,
                "best_policy_state_dict": best_state_dict,
            }
            results[parameter_key].append(record)
            print(
                f"{label} {algorithm_name} p={parameter_key:g} run={run_idx} "
                f"value={record['final_value']:.6g} transitions={record['total_simulator_transitions']} "
                f"runtime={format_runtime(record['runtime_seconds'])}"
            )
    return results


def run_cybersecurity_scenario(
    name: str,
    config: CybersecurityConfig,
    simplex_lambdas: Sequence[float],
    logit_epsilons: Sequence[float],
    budgets: Dict[str, Dict[str, int]],
    run_plans: List[Dict[str, object]],
    flow_mode: str,
    flow_particles: int = 0,
    train_horizon: Optional[int] = None,
    validation_horizon: Optional[int] = None,
    show_progress: bool = True,
    early_stopping_patience: Optional[int] = None,
    max_runtime_seconds: Optional[float] = None,
) -> Dict[str, object]:
    train_horizon = config.T_train if train_horizon is None else int(train_horizon)
    validation_horizon = config.T_val if validation_horizon is None else int(validation_horizon)
    costs = simulator_cost_table(train_horizon, flow_mode, budgets, flow_particles=flow_particles)
    display(costs.round(4))
    simplex_results = train_cybersecurity_method(
        "simplex",
        config,
        simplex_lambdas,
        budgets["Simplex"],
        run_plans,
        flow_mode,
        flow_particles,
        train_horizon,
        validation_horizon,
        label=name,
        show_progress=show_progress,
        early_stopping_patience=early_stopping_patience,
        max_runtime_seconds=max_runtime_seconds,
    )
    logits_results = train_cybersecurity_method(
        "logits",
        config,
        logit_epsilons,
        budgets["Logits"],
        run_plans,
        flow_mode,
        flow_particles,
        train_horizon,
        validation_horizon,
        label=name,
        show_progress=show_progress,
        early_stopping_patience=early_stopping_patience,
        max_runtime_seconds=max_runtime_seconds,
    )
    return {
        "name": name,
        "flow_mode": flow_mode,
        "flow_particles": flow_particles if flow_mode == "particle" else 0,
        "budgets": copy.deepcopy(budgets),
        "costs": costs,
        "train_horizon": train_horizon,
        "validation_horizon": validation_horizon,
        "results": {"Simplex": simplex_results, "Logits": logits_results},
    }


## Reporting And Plotting


In [7]:
def metric_from_run(run: Dict[str, object], key: str) -> float:
    if key in run.get("reference_metrics", {}):
        return float(run["reference_metrics"][key])
    return float(run[key])


def summarize_metric_rows(result_groups, metric_keys: Sequence[str]) -> pd.DataFrame:
    rows = []
    for algorithm_name, result_group in result_groups.items():
        for perturbation, runs in result_group.items():
            row = {"algorithm": algorithm_name, "parameter": float(perturbation), "runs": len(runs)}
            ddof = std_ddof(len(runs))
            for key in metric_keys:
                values = np.asarray([metric_from_run(run, key) for run in runs], dtype=float)
                row[f"{key}_mean"] = float(values.mean())
                row[f"{key}_std"] = float(values.std(ddof=ddof))
            for key in [
                "runtime_seconds",
                "episodes_completed",
                "main_trajectories",
                "auxiliary_trajectories",
                "simulator_transitions_per_update",
                "total_simulator_transitions",
                "best_validation_value",
            ]:
                if key in runs[0]:
                    row[f"{key}_mean"] = float(np.mean([run[key] for run in runs]))
            rows.append(row)
    return pd.DataFrame(rows).set_index(["algorithm", "parameter"]).sort_index()


def interpolate_metric(x: np.ndarray, y: np.ndarray, target_x: np.ndarray) -> np.ndarray:
    order = np.argsort(x)
    x = np.asarray(x, dtype=float)[order]
    y = np.asarray(y, dtype=float)[order]
    unique_x, inverse = np.unique(x, return_inverse=True)
    if len(unique_x) != len(x):
        y_accum = np.zeros_like(unique_x, dtype=float)
        counts = np.zeros_like(unique_x, dtype=float)
        for idx, value in zip(inverse, y):
            y_accum[idx] += value
            counts[idx] += 1
        x = unique_x
        y = y_accum / counts
    return np.interp(target_x, x, y, left=np.nan, right=np.nan)


def matched_radius_summary(result_groups, calibration: pd.DataFrame, metric_key: str = "value", num_points: int = 7):
    rows = []
    for algorithm_name, result_group in result_groups.items():
        if algorithm_name not in {"Simplex", "Logits"}:
            continue
        for perturbation, runs in result_group.items():
            radius_row = calibration[
                (calibration["algorithm"] == algorithm_name)
                & np.isclose(calibration["parameter"], float(perturbation))
            ]
            if radius_row.empty:
                continue
            values = np.asarray([metric_from_run(run, metric_key) for run in runs], dtype=float)
            rows.append(
                {
                    "algorithm": algorithm_name,
                    "parameter": float(perturbation),
                    "radius": float(radius_row.iloc[0]["tv_mean"]),
                    f"{metric_key}_mean": values.mean(),
                    f"{metric_key}_std": values.std(ddof=std_ddof(len(values))),
                }
            )
    native = pd.DataFrame(rows).sort_values(["algorithm", "radius"])
    if native.empty or native["algorithm"].nunique() < 2:
        return pd.DataFrame(), native
    bounds = native.groupby("algorithm")["radius"].agg(["min", "max"])
    lo = bounds["min"].max()
    hi = bounds["max"].min()
    if lo > hi:
        return pd.DataFrame(), native
    radius_points = np.linspace(lo, hi, num_points)
    matched_rows = []
    for algorithm_name, group in native.groupby("algorithm"):
        estimates = interpolate_metric(group["radius"].to_numpy(), group[f"{metric_key}_mean"].to_numpy(), radius_points)
        for radius, estimate in zip(radius_points, estimates):
            matched_rows.append({"algorithm": algorithm_name, "matched_tv_radius": radius, f"{metric_key}_mean_interp": estimate})
    return pd.DataFrame(matched_rows).set_index(["algorithm", "matched_tv_radius"]), native.set_index(["algorithm", "parameter"])


LINE_STYLES = {
    "Simplex": "-",
    "Logits": "--",
    "Finite Adaptive Simplex": "-.",
    "Consistent Adaptive Simplex": ":",
}


def plot_validation_groups(result_groups, title: str):
    fig, ax = plt.subplots(figsize=(10.5, 5.2))
    keys = [(algorithm_name, parameter) for algorithm_name, group in result_groups.items() for parameter in group]
    cmap = plt.get_cmap("tab20", max(len(keys), 1))
    colors = {key: cmap(idx) for idx, key in enumerate(keys)}
    for algorithm_name, result_group in result_groups.items():
        for perturbation, runs in result_group.items():
            episodes, values = aligned_validation_history(runs)
            mean = values.mean(axis=0)
            std = values.std(axis=0, ddof=std_ddof(len(runs)))
            color = colors[(algorithm_name, perturbation)]
            ax.plot(
                episodes,
                mean,
                linestyle=LINE_STYLES.get(algorithm_name, "-"),
                color=color,
                label=f"{algorithm_name}, p={perturbation:g}",
            )
            ax.fill_between(episodes, mean - std, mean + std, color=color, alpha=0.12)
    ax.set_title(title)
    ax.set_xlabel("Training iteration")
    ax.set_ylabel("Exact validation value (mean +/- std. dev.)")
    ax.grid(alpha=0.25)
    ax.legend(ncol=2, fontsize=8)
    fig.tight_layout()
    plt.show()


def best_parameter_for_group(result_group: Dict[float, List[Dict[str, object]]]) -> float:
    rows = []
    for parameter, runs in result_group.items():
        values = np.asarray([run["final_value"] for run in runs], dtype=float)
        rows.append(
            {
                "parameter": float(parameter),
                "value_mean": values.mean(),
                "value_std": values.std(ddof=std_ddof(len(values))),
            }
        )
    return float(pd.DataFrame(rows).sort_values(["value_mean", "value_std", "parameter"], ascending=[False, True, True]).iloc[0]["parameter"])


def plot_best_population_flows(result_groups, title: str):
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
    for algorithm_name, result_group in result_groups.items():
        parameter = best_parameter_for_group(result_group)
        runs = result_group[parameter]
        flows = np.asarray([run["reference_metrics"]["flow"].numpy() for run in runs], dtype=float)
        mean_flow = flows.mean(axis=0)
        infected = mean_flow[:, 0] + mean_flow[:, 2]
        defended = mean_flow[:, 0] + mean_flow[:, 1]
        times = np.arange(mean_flow.shape[0])
        label = f"{algorithm_name}, p={parameter:g}"
        axes[0].plot(times, infected, label=label)
        axes[1].plot(times, defended, label=label)
    axes[0].set_title("Infected fraction")
    axes[1].set_title("Defended fraction")
    for ax in axes:
        ax.set_xlabel("Validation time")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def scenario_summary_table(scenario: Dict[str, object], experiment: str, horizon: Optional[int] = None) -> pd.DataFrame:
    table = summarize_metric_rows(
        scenario["results"],
        [
            "value",
            "terminal_infected",
            "terminal_defended",
            "mean_infected",
            "mean_defended",
            "mean_update_probability",
        ],
    ).reset_index()
    table.insert(0, "experiment", experiment)
    table.insert(1, "flow_mode", scenario["flow_mode"])
    table.insert(2, "T_train", int(scenario["train_horizon"] if horizon is None else horizon))
    table.insert(3, "flow_particles", int(scenario["flow_particles"]))
    if "costs" in scenario and "simulator_transitions_per_update" in scenario["costs"].columns:
        cost_lookup = scenario["costs"]["simulator_transitions_per_update"].to_dict()
        table["scenario_cost_per_update"] = table["algorithm"].map(cost_lookup)
    elif "costs" in scenario and "expected_transitions_per_update" in scenario["costs"].columns:
        cost_lookup = scenario["costs"]["expected_transitions_per_update"].to_dict()
        table["scenario_cost_per_update"] = table["algorithm"].map(cost_lookup)
    leading_columns = ["experiment", "flow_mode", "T_train", "flow_particles", "algorithm", "parameter"]
    return table[leading_columns + [column for column in table.columns if column not in leading_columns]]


def show_cybersecurity_results(scenario: Dict[str, object], calibration: pd.DataFrame):
    result_groups = scenario["results"]
    print(scenario["name"])
    display(scenario["costs"].round(4))
    display(calibration.round(4))
    matched, native = matched_radius_summary(result_groups, calibration, metric_key="value")
    display(native.round(6))
    display(matched.round(6))
    plot_validation_groups(result_groups, f"{scenario['name']}: validation value")
    display(
        summarize_metric_rows(
            result_groups,
            [
                "value",
                "terminal_DI",
                "terminal_DS",
                "terminal_UI",
                "terminal_US",
                "terminal_infected",
                "terminal_defended",
                "mean_update_probability",
            ],
        ).round(6)
    )
    plot_best_population_flows(result_groups, f"{scenario['name']}: best population flows")


def best_fixed_parameters(scenario: Dict[str, object], calibration: pd.DataFrame) -> Dict[str, float]:
    summary = summarize_metric_rows(scenario["results"], ["value"]).reset_index()
    radius_lookup = {
        (row["algorithm"], float(row["parameter"])): float(row["tv_mean"])
        for _, row in calibration.iterrows()
    }
    summary["radius"] = [
        radius_lookup.get((row["algorithm"], float(row["parameter"])), float("inf"))
        for _, row in summary.iterrows()
    ]
    selected = {}
    for algorithm_name in ["Simplex", "Logits"]:
        group = summary[summary["algorithm"] == algorithm_name].copy()
        group = group.sort_values(["value_mean", "value_std", "radius"], ascending=[False, True, True])
        selected[algorithm_name] = float(group.iloc[0]["parameter"])
    return selected


def controller_diagnostics_frame(scenario: Dict[str, object]) -> pd.DataFrame:
    rows = []
    for algorithm_name, parameter_groups in scenario["results"].items():
        for parameter, runs in parameter_groups.items():
            for run in runs:
                for record in run.get("controller_diagnostics", []):
                    row = {
                        "algorithm": algorithm_name,
                        "parameter": parameter,
                        "run_idx": run["run_idx"],
                    }
                    row.update(record)
                    rows.append(row)
    return pd.DataFrame(rows)


## Exact-Gradient Diagnostics


In [8]:
def exact_gradient(
    env: CybersecurityMFC,
    policy: CybersecurityPolicy,
    mu0: torch.Tensor,
    horizon: int,
) -> Tuple[torch.Tensor, torch.Tensor]:
    for parameter in policy.parameters():
        parameter.requires_grad_(True)
    value = env.exact_value(policy, mu0, horizon)
    grads = torch.autograd.grad(value, tuple(policy.parameters()), allow_unused=False)
    flat_grad = torch.cat([grad.detach().reshape(-1) for grad in grads])
    return value.detach(), flat_grad


def cosine_similarity_flat(x: torch.Tensor, y: torch.Tensor) -> float:
    denom = torch.linalg.norm(x) * torch.linalg.norm(y)
    if float(denom.item()) == 0.0:
        return float("nan")
    return float((x.flatten() @ y.flatten() / denom).item())


def gradient_diagnostic_summary(
    config: CybersecurityConfig,
    payload: Dict[str, object],
    algorithm_name: str,
    perturbation: float,
    budget: Dict[str, int],
    mu0: torch.Tensor,
    horizon: int,
    flow_mode: str,
    flow_particles: int,
    repetitions: int,
    label: str,
    show_progress: bool = True,
) -> Dict[str, object]:
    env = CybersecurityMFC(config)
    policy = load_policy(config, payload, trainable=True)
    mu0 = mu0.to(dtype=config.dtype, device=config.device)
    B = int(budget["B"])
    n_aux_or_inner = int(budget["n"])
    _, oracle_grad = exact_gradient(env, policy, mu0, horizon)
    per_estimate_transitions = simulator_transitions_per_update(
        algorithm_name,
        horizon,
        B,
        n_aux_or_inner,
        flow_mode=flow_mode,
        flow_particles=flow_particles,
    )

    samples = []
    iterator = range(repetitions)
    if show_progress:
        iterator = tqdm(iterator, desc=f"{label} {algorithm_name} p={perturbation:g}", leave=False)
    algorithm = SimplexPerturbedMFREINFORCE(env) if algorithm_name == "Simplex" else LogitsPerturbedMFREINFORCE(env)

    for _ in iterator:
        mu_flow = training_population_flow(env, algorithm, policy, mu0, horizon, flow_mode, flow_particles)
        if algorithm_name == "Simplex":
            grad_hat, _ = algorithm.complete_gradient_estimate(
                policy,
                mu_flow,
                float(perturbation),
                B,
                n_aux_or_inner,
                eta=float(perturbation),
                baseline="batch_mean",
            )
        else:
            grad_hat, _ = algorithm.gradient_estimate(
                policy,
                mu0,
                float(perturbation),
                B,
                n_aux_or_inner,
                flow_particles=1,
                horizon=horizon,
                mu_flow=mu_flow,
            )
        samples.append(grad_hat.detach().reshape(-1))

    samples_t = torch.stack(samples)
    mean_grad = samples_t.mean(dim=0)
    bias = mean_grad - oracle_grad
    covariance_trace = float(samples_t.var(dim=0, unbiased=repetitions > 1).sum().item()) if repetitions > 1 else 0.0
    mse = ((samples_t - oracle_grad.unsqueeze(0)).square().sum(dim=1)).mean()
    return {
        "label": label,
        "algorithm": algorithm_name,
        "flow_mode": flow_mode,
        "flow_particles": flow_particles if flow_mode == "particle" else 0,
        "parameter": float(perturbation),
        "repetitions": repetitions,
        "B": B,
        "n_aux_or_inner": n_aux_or_inner,
        "oracle_grad_norm": float(torch.linalg.norm(oracle_grad).item()),
        "mean_grad_norm": float(torch.linalg.norm(mean_grad).item()),
        "bias_norm": float(torch.linalg.norm(bias).item()),
        "covariance_trace": covariance_trace,
        "mse": float(mse.item()),
        "cosine_to_oracle": cosine_similarity_flat(mean_grad, oracle_grad),
        "simulator_transitions_per_estimate": int(per_estimate_transitions),
        "simulator_transitions": int(per_estimate_transitions * repetitions),
    }


def run_cybersecurity_diagnostics(
    scenario: Dict[str, object],
    config: CybersecurityConfig,
    simplex_lambdas: Sequence[float],
    logit_epsilons: Sequence[float],
    run_plans: List[Dict[str, object]],
    repetitions: int = 8,
    show_progress: bool = True,
) -> pd.DataFrame:
    results = scenario["results"]
    best_simplex = best_parameter_for_group(results["Simplex"])
    best_logits = best_parameter_for_group(results["Logits"])
    controls = [
        ("initial", run_plans[0]["initial_control"]),
        (f"simplex_final_lambda_{best_simplex:g}", payload_from_record(results["Simplex"][best_simplex][0])),
        (f"logits_final_epsilon_{best_logits:g}", payload_from_record(results["Logits"][best_logits][0])),
    ]
    rows = []
    mu0 = fixed_validation_law(config)
    for label, payload in controls:
        for perturbation in simplex_lambdas:
            rows.append(
                gradient_diagnostic_summary(
                    config,
                    payload,
                    "Simplex",
                    perturbation,
                    scenario["budgets"]["Simplex"],
                    mu0,
                    scenario["train_horizon"],
                    scenario["flow_mode"],
                    scenario["flow_particles"],
                    repetitions,
                    label,
                    show_progress=show_progress,
                )
            )
        for perturbation in logit_epsilons:
            rows.append(
                gradient_diagnostic_summary(
                    config,
                    payload,
                    "Logits",
                    perturbation,
                    scenario["budgets"]["Logits"],
                    mu0,
                    scenario["train_horizon"],
                    scenario["flow_mode"],
                    scenario["flow_particles"],
                    repetitions,
                    label,
                    show_progress=show_progress,
                )
            )
    return pd.DataFrame(rows)


## Adaptive Exact-Flow Comparison


In [9]:
def run_adaptive_exact_comparison(
    name: str,
    config: CybersecurityConfig,
    equal_budget_exact_scenario: Dict[str, object],
    calibration: pd.DataFrame,
    run_plans: List[Dict[str, object]],
    adaptive_initial_lambdas: Sequence[float],
    budgets: Dict[str, Dict[str, int]],
    show_progress: bool = True,
    max_runtime_seconds: Optional[float] = None,
) -> Dict[str, object]:
    best_fixed = best_fixed_parameters(equal_budget_exact_scenario, calibration)
    train_horizon = int(equal_budget_exact_scenario["train_horizon"])
    validation_horizon = int(equal_budget_exact_scenario["validation_horizon"])
    finite_results = train_cybersecurity_method(
        "finite_adaptive",
        config,
        adaptive_initial_lambdas,
        budgets["Simplex"],
        run_plans,
        flow_mode="exact",
        flow_particles=0,
        train_horizon=train_horizon,
        validation_horizon=validation_horizon,
        label=name,
        show_progress=show_progress,
        max_runtime_seconds=max_runtime_seconds,
    )
    consistent_results = train_cybersecurity_method(
        "consistent_adaptive",
        config,
        adaptive_initial_lambdas,
        budgets["Simplex"],
        run_plans,
        flow_mode="exact",
        flow_particles=0,
        train_horizon=train_horizon,
        validation_horizon=validation_horizon,
        label=name,
        show_progress=show_progress,
        max_runtime_seconds=max_runtime_seconds,
    )
    controller_config = finite_controller_for_initial_lambda(adaptive_initial_lambdas[0])
    costs = adaptive_cost_table(train_horizon, "exact", budgets, controller_config, flow_particles=0)
    return {
        "name": name,
        "flow_mode": "exact",
        "flow_particles": 0,
        "budgets": copy.deepcopy(budgets),
        "costs": costs,
        "best_fixed_parameters": best_fixed,
        "train_horizon": train_horizon,
        "validation_horizon": validation_horizon,
        "results": {
            "Simplex": {
                best_fixed["Simplex"]: equal_budget_exact_scenario["results"]["Simplex"][best_fixed["Simplex"]]
            },
            "Logits": {
                best_fixed["Logits"]: equal_budget_exact_scenario["results"]["Logits"][best_fixed["Logits"]]
            },
            "Finite Adaptive Simplex": finite_results,
            "Consistent Adaptive Simplex": consistent_results,
        },
    }


def show_adaptive_results(scenario: Dict[str, object]):
    print(scenario["name"])
    print("Best fixed parameters:", scenario["best_fixed_parameters"])
    display(scenario["costs"].round(4))
    display(
        summarize_metric_rows(
            scenario["results"],
            [
                "value",
                "terminal_infected",
                "terminal_defended",
                "mean_update_probability",
            ],
        ).round(6)
    )
    plot_validation_groups(scenario["results"], f"{scenario['name']}: validation value")
    plot_best_population_flows(scenario["results"], f"{scenario['name']}: best population flows")
    diagnostics = controller_diagnostics_frame(scenario)
    if not diagnostics.empty:
        display(
            diagnostics[
                [
                    "algorithm",
                    "parameter",
                    "run_idx",
                    "episode",
                    "lambda_plus",
                    "lambda_minus",
                    "bias_proxy_sq",
                    "variance_proxy",
                    "signed_pressure",
                    "directional_cosine",
                    "direction_triggered",
                ]
            ].round(6)
        )


## Common Experiment Setup


In [10]:
### USING LIGHTER CONFIG FOR TESTING
config = CybersecurityConfig(
    device=device,
    dtype=DTYPE,
    T_train=3,
    T_val=50,
    hidden_units=32,
    lr=1e-3,
    n_train=10_000,
    training_runs=3,
    validate_every=10,
)
train_horizon = config.T_train
validation_horizon = config.T_val

simplex_lambdas = [0.1, 0.2, 0.4]
logit_epsilons = [0.2, 0.5, 1.0]
adaptive_initial_lambdas = [0.1, 0.15, 0.2]

same_parameter_budgets = {
    "Simplex": {"B": 200, "n": 10},
    "Logits": {"B": 200, "n": 10},
}
equal_simulator_budgets = simplex_budget_matching_logit(train_horizon, logit_B=200, logit_n=10)
particle_flow_particles = 200

early_stopping_patience = None
max_runtime_seconds = None
diagnostic_repetitions = 8
perturbation_calibration_samples = 20_000
reference_law_count = 20_000

cybersecurity_run_plans = prepare_paired_run_plans(config, seed_base=51_000)
cybersecurity_reference_laws = collect_reference_laws(
    config,
    cybersecurity_run_plans[0]["initial_control"],
    reference_law_count,
    train_horizon,
)
cybersecurity_calibration = calibrate_perturbation_radii(
    config,
    cybersecurity_reference_laws,
    simplex_lambdas,
    logit_epsilons,
    num_samples=perturbation_calibration_samples,
)

exact_same_parameter_costs = simulator_cost_table(train_horizon, "exact", same_parameter_budgets)
exact_equal_budget_costs = simulator_cost_table(train_horizon, "exact", equal_simulator_budgets)
particle_equal_budget_costs = simulator_cost_table(
    train_horizon,
    "particle",
    equal_simulator_budgets,
    flow_particles=particle_flow_particles,
)

display(cybersecurity_calibration.round(4))
print("Exact flow, same parameters")
display(exact_same_parameter_costs.round(4))
print("Exact flow, equal simulator budget")
display(exact_equal_budget_costs.round(4))
print("Estimated particle flow, equal simulator budget")
display(particle_equal_budget_costs.round(4))


,algorithm,parameter,tv_mean,tv_median,tv_std,tv_p10,tv_p90
0,Simplex,0.1,0.0354,0.0341,0.0154,0.0162,0.0564
1,Simplex,0.2,0.0708,0.0683,0.0308,0.0325,0.1128
2,Simplex,0.4,0.1416,0.1365,0.0616,0.0649,0.2257
3,Logits,0.2,0.0622,0.0580,0.0320,0.0246,0.1058
4,Logits,0.5,0.1513,0.1420,0.0761,0.0609,0.2552
5,Logits,1.0,0.2779,0.2653,0.1308,0.1166,0.4574


Exact flow, same parameters


,B,n,horizon,flow_mode,flow_particles,simulator_transitions_per_update,ratio_vs_simplex
algorithm,,,,,,,
Simplex,200,10,3,exact,0,630,1.0
Logits,200,10,3,exact,0,12600,20.0


Exact flow, equal simulator budget


,B,n,horizon,flow_mode,flow_particles,simulator_transitions_per_update,ratio_vs_simplex
algorithm,,,,,,,
Simplex,3675,525,3,exact,0,12600,1.0
Logits,200,10,3,exact,0,12600,1.0


Estimated particle flow, equal simulator budget


,B,n,horizon,flow_mode,flow_particles,simulator_transitions_per_update,ratio_vs_simplex
algorithm,,,,,,,
Simplex,3675,525,3,particle,200,13200,1.0
Logits,200,10,3,particle,200,13200,1.0


## Experiment 1: Exact Flow, Equal Parameters

Both estimators use `(B, n) = (200, 10)`. This exposes the raw simulator-cost gap between simplex and logit perturbations while keeping nominal Monte Carlo settings identical.


In [11]:
exact_equal_parameters = run_cybersecurity_scenario(
    "equal_parameters_exact",
    config,
    simplex_lambdas,
    logit_epsilons,
    same_parameter_budgets,
    cybersecurity_run_plans,
    flow_mode="exact",
    train_horizon=train_horizon,
    validation_horizon=validation_horizon,
    show_progress=True,
    early_stopping_patience=early_stopping_patience,
    max_runtime_seconds=max_runtime_seconds,
)


,B,n,horizon,flow_mode,flow_particles,simulator_transitions_per_update,ratio_vs_simplex
algorithm,,,,,,,
Simplex,200,10,3,exact,0,630,1.0
Logits,200,10,3,exact,0,12600,20.0


equal_parameters_exact Simplex p=0.1 run=0: 100%|██████████| 10000/10000 [08:27<00:00, 19.70it/s, grad=2.42, value=-0.1589]   


equal_parameters_exact Simplex p=0.1 run=0 value=-0.158921 transitions=6300000 runtime=8m 27s


equal_parameters_exact Simplex p=0.1 run=1: 100%|██████████| 10000/10000 [09:53<00:00, 16.85it/s, grad=131, value=-0.1598]    


equal_parameters_exact Simplex p=0.1 run=1 value=-0.159828 transitions=6300000 runtime=9m 53s


...

equal_parameters_exact Logits p=1 run=0 value=-0.151697 transitions=126000000 runtime=1h 42m 47s


equal_parameters_exact Logits p=1 run=1: 100%|██████████| 10000/10000 [1:42:37<00:00,  1.62it/s, grad=0.00039, value=-0.1517]


equal_parameters_exact Logits p=1 run=1 value=-0.151687 transitions=126000000 runtime=1h 42m 37s


equal_parameters_exact Logits p=1 run=2:  12%|█▏        | 1219/10000 [14:07<1:41:44,  1.44it/s, grad=0.0167, value=-0.1521] 


KeyboardInterrupt: 

In [13]:
exact_equal_parameters_gradient_study = run_cybersecurity_diagnostics(
    exact_equal_parameters,
    config,
    simplex_lambdas,
    logit_epsilons,
    cybersecurity_run_plans,
    repetitions=diagnostic_repetitions,
)
display(exact_equal_parameters_gradient_study.round(6))
show_cybersecurity_results(exact_equal_parameters, cybersecurity_calibration)


NameError: name 'exact_equal_parameters' is not defined

## Experiment 2: Exact Flow, Equal Simulator Budget

Logits uses `(B, n) = (200, 10)`. Simplex uses `(B, n) = (3675, 525)`, so both methods use `12600` simulated transitions per update for `T_train=3`.


In [ ]:
exact_equal_budget = run_cybersecurity_scenario(
    "equal_budget_exact",
    config,
    simplex_lambdas,
    logit_epsilons,
    equal_simulator_budgets,
    cybersecurity_run_plans,
    flow_mode="exact",
    train_horizon=train_horizon,
    validation_horizon=validation_horizon,
    show_progress=True,
    early_stopping_patience=early_stopping_patience,
    max_runtime_seconds=max_runtime_seconds,
)


In [ ]:
exact_equal_budget_gradient_study = run_cybersecurity_diagnostics(
    exact_equal_budget,
    config,
    simplex_lambdas,
    logit_epsilons,
    cybersecurity_run_plans,
    repetitions=diagnostic_repetitions,
)
display(exact_equal_budget_gradient_study.round(6))
show_cybersecurity_results(exact_equal_budget, cybersecurity_calibration)


## Experiment 3: Estimated Flow, Equal Simulator Budget

Both estimators condition on a particle-estimated training population flow with `flow_particles=200`. Validation remains deterministic and exact.


In [ ]:
estimated_equal_budget = run_cybersecurity_scenario(
    "equal_budget_estimated_flow",
    config,
    simplex_lambdas,
    logit_epsilons,
    equal_simulator_budgets,
    cybersecurity_run_plans,
    flow_mode="particle",
    flow_particles=particle_flow_particles,
    train_horizon=train_horizon,
    validation_horizon=validation_horizon,
    show_progress=True,
    early_stopping_patience=early_stopping_patience,
    max_runtime_seconds=max_runtime_seconds,
)


In [ ]:
estimated_equal_budget_gradient_study = run_cybersecurity_diagnostics(
    estimated_equal_budget,
    config,
    simplex_lambdas,
    logit_epsilons,
    cybersecurity_run_plans,
    repetitions=diagnostic_repetitions,
)
display(estimated_equal_budget_gradient_study.round(6))
show_cybersecurity_results(estimated_equal_budget, cybersecurity_calibration)


## Final Experiment: Adaptive Exact-Flow Comparison

The best fixed simplex and logits configurations are selected from the equal-budget exact-flow experiment by highest mean final exact validation value, then lower standard deviation, then smaller calibrated perturbation radius. Adaptive simplex methods use the same base `(B, n)` budget as equal-budget simplex; controller diagnostic transitions are tracked separately in each run.


In [ ]:
adaptive_exact_comparison = run_adaptive_exact_comparison(
    "adaptive_exact_vs_best_fixed",
    config,
    exact_equal_budget,
    cybersecurity_calibration,
    cybersecurity_run_plans,
    adaptive_initial_lambdas,
    equal_simulator_budgets,
    show_progress=True,
    max_runtime_seconds=max_runtime_seconds,
)
show_adaptive_results(adaptive_exact_comparison)


## Final Summary

The tables below collect experiment-level summaries, gradient diagnostics, and adaptive-controller diagnostics. They assume all experiment cells above have been executed.


In [ ]:
cross_protocol_summary = pd.concat(
    [
        scenario_summary_table(exact_equal_parameters, "equal_parameters_exact"),
        scenario_summary_table(exact_equal_budget, "equal_budget_exact"),
        scenario_summary_table(estimated_equal_budget, "equal_budget_estimated_flow"),
        scenario_summary_table(adaptive_exact_comparison, "adaptive_exact_vs_best_fixed"),
    ],
    ignore_index=True,
)

all_gradient_diagnostics = pd.concat(
    [
        exact_equal_parameters_gradient_study.assign(experiment="equal_parameters_exact"),
        exact_equal_budget_gradient_study.assign(experiment="equal_budget_exact"),
        estimated_equal_budget_gradient_study.assign(experiment="equal_budget_estimated_flow"),
    ],
    ignore_index=True,
)

adaptive_controller_diagnostics = controller_diagnostics_frame(adaptive_exact_comparison)

display(cross_protocol_summary.round(6))
display(all_gradient_diagnostics.round(6))
display(adaptive_controller_diagnostics.round(6))
